# 07 Test the semantic model

Asks the model **WindFarm Operations** and the Lakehouse the same questions and compares the answers: a DAX query
with the model's measures on one side, a Spark SQL query on the tables of notebook 05 on the other. The
questions are those of the question bank that a measure can answer; the heading of each check names them.

A check fails when a measure, a relationship or the model's copy of the data is wrong. Run it after
notebooks 05 and 06, and after every change of a measure. The first cell installs the library
semantic-link-labs for this session; every notebook has its own session.

In [ ]:
%pip install -q semantic-link-labs==0.15.2

In [ ]:
# Generated by tools/build-phase2.py from tools/model_spec.py. Change it there.
def acceptance_checks(d):
    """Checks for notebook 07. d holds dates: as_of, d90, month, m6, m12, year, y12 (see the notebook).
    Each check runs a DAX query on the model and a Spark SQL query on the Lakehouse tables; the two
    must agree. Columns have the same names on both sides; keys lists the grouping columns."""
    def window(start, end):          # dates from start (included) to end (excluded), as a DAX filter
        return (f"FILTER(ALL('Date'[Date]), 'Date'[Date] >= DATE({start.year}, {start.month}, {start.day}) "
                f"&& 'Date'[Date] < DATE({end.year}, {end.month}, {end.day}))")
    end = d["end"]                   # the day after the last day with data
    down = "('Forced outage', 'Scheduled maintenance', 'Planned corrective action')"
    own = "('Component fault', 'Scheduled maintenance', 'Corrective maintenance', 'Technical derate')"
    return [
        {"id": "SM-01, SM-02", "title": "Energy, budget and capacity factor by month, last 12 whole months",
         "keys": ["YearMonth"],
         "dax": f"""EVALUATE SUMMARIZECOLUMNS('Date'[YearMonth], {window(d['m12'], d['month'])},
                 "Energy", [Energy MWh], "BudgetP50", [Budget P50 MWh], "CapacityFactor", [Capacity Factor])""",
         "sql": f"""SELECT d.YearMonth, sum(s.EnergyKwh) / 1000 AS Energy, max(b.P50EnergyMwh) AS BudgetP50,
                           sum(s.EnergyKwh) / sum(t.RatedPowerKw / 6.0) AS CapacityFactor
                    FROM fact_scada_10min s JOIN dim_date d ON d.Date = s.Date
                    JOIN dim_turbine t ON t.TurbineId = s.TurbineId
                    JOIN fact_monthly_budget b ON b.YearMonth = d.YearMonth
                    WHERE s.Date >= DATE'{d['m12']}' AND s.Date < DATE'{d['month']}' GROUP BY d.YearMonth"""},
        {"id": "SM-03", "title": "Availability of each turbine, last 90 days", "keys": ["TurbineId"],
         "dax": f"""EVALUATE SUMMARIZECOLUMNS('Turbine'[TurbineId], {window(d['d90'], end)},
                 "TimeAvailability", [Time Availability], "EnergyAvailability", [Energy Availability],
                 "ForcedOutageHours", [Forced Outage Hours])""",
         "sql": f"""SELECT TurbineId,
                           1 - sum(CASE WHEN AvailabilityCategory IN {down} THEN 1 ELSE 0 END) / count(*) AS TimeAvailability,
                           sum(EnergyKwh) / (sum(EnergyKwh) + sum(CASE WHEN LossCause IN {own} THEN LostEnergyKwh ELSE 0 END)) AS EnergyAvailability,
                           nullif(sum(CASE WHEN AvailabilityCategory = 'Forced outage' THEN 1 ELSE 0 END), 0) / 6.0 AS ForcedOutageHours
                    FROM fact_scada_10min
                    WHERE Date >= DATE'{d['d90']}' AND AvailabilityCategory <> 'Information unavailable' GROUP BY TurbineId"""},
        {"id": "PE-01", "title": "Performance index of each turbine, last 90 days", "keys": ["TurbineId"],
         "dax": f"""EVALUATE SUMMARIZECOLUMNS('Turbine'[TurbineId], {window(d['d90'], end)},
                 "PerformanceIndex", [Performance Index])""",
         "sql": f"""SELECT TurbineId, sum(EnergyKwh) / sum(PossibleEnergyKwh) AS PerformanceIndex
                    FROM fact_scada_10min
                    WHERE Date >= DATE'{d['d90']}' AND AvailabilityCategory = 'Full performance' GROUP BY TurbineId"""},
        {"id": "SM-04", "title": "Lost energy by cause and its value, last 12 months", "keys": ["LossCause"],
         "dax": f"""EVALUATE SUMMARIZECOLUMNS('Scada'[LossCause], {window(d['y12'], end)},
                 "LostEnergy", [Lost Energy MWh], "Value", [Lost Energy Value USD])""",
         "sql": f"""SELECT s.LossCause, sum(s.LostEnergyKwh) / 1000 AS LostEnergy,
                           sum(s.LostEnergyKwh) / 1000 * max(st.PpaPriceUsdPerMwh) AS Value
                    FROM fact_scada_10min s CROSS JOIN dim_site st
                    WHERE s.Date >= DATE'{d['y12']}' AND s.LostEnergyKwh > 0 GROUP BY s.LossCause"""},
        {"id": "AM-02", "title": "Curtailment share, deemed and unpaid curtailment, last 12 months", "keys": [],
         "dax": f"""EVALUATE CALCULATETABLE(ROW("CurtailmentShare", [Curtailment Share], "Deemed", [Deemed Generation MWh],
                 "Unpaid", [Unpaid Curtailment MWh]), {window(d['y12'], end)})""",
         "sql": f"""SELECT (SELECT sum(CASE WHEN LossCause = 'Grid curtailment' THEN LostEnergyKwh ELSE 0 END)
                                   / (sum(EnergyKwh) + sum(LostEnergyKwh))
                            FROM fact_scada_10min WHERE Date >= DATE'{d['y12']}') AS CurtailmentShare,
                           sum(CASE WHEN Compensation = 'Paid' THEN LostEnergyMwh END) AS Deemed,
                           sum(CASE WHEN Compensation = 'Not paid' THEN LostEnergyMwh END) AS Unpaid
                    FROM fact_curtailments WHERE StartDate >= DATE'{d['y12']}'"""},
        {"id": "RE-02", "title": "Failures and mean time to repair of each turbine, last 12 months",
         "keys": ["TurbineId"],
         "dax": f"""EVALUATE SUMMARIZECOLUMNS('Turbine'[TurbineId], {window(d['y12'], end)},
                 "Failures", [Failures], "DowntimeHours", [Failure Downtime Hours], "MTTR", [MTTR Hours])""",
         "sql": f"""SELECT TurbineId, count(*) AS Failures, sum(DurationHours) AS DowntimeHours, avg(DurationHours) AS MTTR
                    FROM fact_turbine_events
                    WHERE AvailabilityCategory = 'Forced outage' AND Cause = 'Component fault'
                      AND StartDate >= DATE'{d['y12']}' GROUP BY TurbineId"""},
        {"id": "AM-01, AM-07", "title": "Revenue and carbon dioxide avoided, year to date", "keys": [],
         "dax": f"""EVALUATE CALCULATETABLE(ROW("Energy", [Energy MWh], "Revenue", [Revenue USD], "Co2", [CO2 Avoided t]),
                 {window(d['year'], end)})""",
         "sql": f"""SELECT e.Energy, (e.Energy + coalesce(c.Deemed, 0)) * st.PpaPriceUsdPerMwh AS Revenue,
                           e.Energy * st.GridEmissionFactorKgCo2PerMwh / 1000 AS Co2
                    FROM (SELECT sum(EnergyKwh) / 1000 AS Energy FROM fact_scada_10min WHERE Date >= DATE'{d['year']}') e
                    CROSS JOIN (SELECT sum(CASE WHEN Compensation = 'Paid' THEN LostEnergyMwh END) AS Deemed
                                FROM fact_curtailments WHERE StartDate >= DATE'{d['year']}') c
                    CROSS JOIN dim_site st"""},
        {"id": "AM-04", "title": "Repair cost by who paid, year to date", "keys": [],
         "dax": f"""EVALUATE CALCULATETABLE(ROW("Owner", [Repair Cost Paid by Owner USD],
                 "Warranty", [Repair Cost Covered by Warranty USD], "PerMwh", [Repair Cost USD per MWh]),
                 {window(d['year'], end)})""",
         "sql": f"""SELECT w.Owner, w.Warranty, w.Owner / e.Energy AS PerMwh
                    FROM (SELECT sum(CASE WHEN CostBorneBy = 'Owner' THEN TotalCostUsd END) AS Owner,
                                 sum(CASE WHEN CostBorneBy = 'Warranty' THEN TotalCostUsd END) AS Warranty
                          FROM fact_work_orders
                          WHERE WorkType = 'Corrective' AND Status = 'Closed' AND OpenedDate >= DATE'{d['year']}') w
                    CROSS JOIN (SELECT sum(EnergyKwh) / 1000 AS Energy FROM fact_scada_10min WHERE Date >= DATE'{d['year']}') e"""},
        {"id": "MP-02", "title": "Preventive maintenance compliance, work due in the last 12 months", "keys": [],
         "dax": f"""EVALUATE CALCULATETABLE(ROW("PmCompliance", [PM Compliance]), {window(d['y12'], end)})""",
         "sql": f"""SELECT sum(CASE WHEN PmResult = 'Done within a week of the plan' THEN 1 ELSE 0 END) / count(*) AS PmCompliance
                    FROM fact_work_orders WHERE PmResult IS NOT NULL AND ScheduledDate >= DATE'{d['y12']}'"""},
        {"id": "AM-06", "title": "Captured and mean real-time price, last 12 months", "keys": [],
         "dax": f"""EVALUATE CALCULATETABLE(ROW("CapturedPrice", [Captured Price USD per MWh],
                 "MeanPrice", [Mean Real-Time Price USD per MWh]), {window(d['y12'], end)})""",
         "sql": f"""SELECT sum(SiteEnergyMwh * RealTimePriceUsdPerMwh) / sum(SiteEnergyMwh) AS CapturedPrice,
                           avg(RealTimePriceUsdPerMwh) AS MeanPrice
                    FROM fact_market_hourly WHERE Date >= DATE'{d['y12']}'"""},
        {"id": "AM-05", "title": "Day-ahead forecast error by month, last 6 whole months", "keys": ["YearMonth"],
         "dax": f"""EVALUATE SUMMARIZECOLUMNS('Date'[YearMonth], {window(d['m6'], d['month'])},
                 "ForecastError", [Forecast Error])""",
         "sql": f"""SELECT d.YearMonth, avg(f.AbsErrorMw) / max(st.CapacityMw) AS ForecastError
                    FROM fact_forecast_hourly f JOIN dim_date d ON d.Date = f.Date CROSS JOIN dim_site st
                    WHERE f.Date >= DATE'{d['m6']}' AND f.Date < DATE'{d['month']}' GROUP BY d.YearMonth"""},
    ]


In [ ]:
import datetime as dt
import numpy as np
import pandas as pd
import sempy.fabric as fabric
import sempy_labs as labs

MODEL = "WindFarm Operations"
spark.conf.set("spark.sql.session.timeZone", "UTC")
labs.refresh_semantic_model(dataset=MODEL)                 # the model must read what the Lakehouse holds now


def months_back(day, months):
    month = day.year * 12 + day.month - 1 - months
    return dt.date(month // 12, month % 12 + 1, 1)


last_day = spark.sql("SELECT max(Date) FROM fact_scada_10min").first()[0]
D = {"end": last_day + dt.timedelta(days=1), "d90": last_day - dt.timedelta(days=90),
     "month": last_day.replace(day=1), "year": last_day.replace(month=1, day=1),
     "m6": months_back(last_day, 6), "m12": months_back(last_day, 12),
     "y12": months_back(last_day, 12).replace(day=min(last_day.day, 28))}
print("Last day with data:", last_day)

In [ ]:
def tidy(frame, keys):
    frame = frame.copy()
    frame.columns = [c.split("[")[-1].rstrip("]") for c in frame.columns]      # 'Date'[YearMonth] -> YearMonth
    for key in keys:
        frame[key] = frame[key].fillna("(none)").astype(str)
    values = [c for c in frame.columns if c not in keys]
    frame[values] = frame[values].apply(pd.to_numeric, errors="coerce").astype(float)
    return frame


def run_check(check):
    keys = check["keys"]
    model = tidy(pd.DataFrame(fabric.evaluate_dax(MODEL, check["dax"])), keys)
    lake = tidy(spark.sql(check["sql"]).toPandas(), keys)
    if keys:
        both = model.merge(lake, on=keys, how="outer", suffixes=(" model", " lakehouse"))
    else:
        both = model.add_suffix(" model").join(lake.add_suffix(" lakehouse"))
    bad = []
    for column in [c for c in lake.columns if c not in keys]:
        a = both[f"{column} model"].fillna(0.0)
        b = both[f"{column} lakehouse"].fillna(0.0)
        if not np.allclose(a, b, rtol=1e-6, atol=1e-6):
            bad.append(column)
    return both.sort_values(keys) if keys else both, bad


failed = 0
for check in acceptance_checks(D):
    try:
        result, bad = run_check(check)
    except Exception as error:                                # a DAX error counts as a failed check
        result, bad = None, [f"error: {str(error)[:300]}"]
    print(("PASS  " if not bad else "FAIL  ") + check["id"] + ": " + check["title"] + (f"   differs: {bad}" if bad else ""))
    if bad:
        failed += 1
        if result is not None:
            display(result)
print(f"\n{failed} of {len(acceptance_checks(D))} checks failed" if failed else "\nAll checks passed")
assert failed == 0, "The model and the Lakehouse disagree; see the tables above."